# Broker Communities and Target-only Accounts (R2-5.2, R2-5.4)

Separates individual from group-level brokerage and quantifies how the query design shapes the structural roles of institutional accounts:
1. concentration of Diffusion Drivers across communities, with a Fisher z 95% confidence interval for the Spearman correlation between Diffusion Driver count and community size (Section 4.3);
2. community-level brokerage, as unweighted betweenness centrality on the community graph of the largest connected component (Section 4.3, Table S15);
3. accounts that appear only as targets of interactions, and the structural roles of accounts that posted at least one query-matched tweet (Section 4.3).

Inputs: `main-data/Book62-hashed.csv` and, from `derived-data/hashed/`, `actor_roles_classification_hashed.csv`, `network_nodes_with_community_hashed.csv`, `network_edges_hashed.csv`, `Data_with_community_hashed.csv`.

In [1]:
import numpy as np, pandas as pd, networkx as nx
from scipy import stats
roles = pd.read_csv('actor_roles_classification_hashed.csv')                 # derived-data/hashed/
roles['structural_role'] = roles['structural_role'].str.strip()
nodes = pd.read_csv('network_nodes_with_community_hashed.csv', sep=';')       # derived-data/hashed/
edges = pd.read_csv('network_edges_hashed.csv')                               # derived-data/hashed/
tweets = pd.read_csv('Data_with_community_hashed.csv', sep=';', low_memory=False,
                     usecols=['narrative', 'community'])                      # derived-data/hashed/
authors = pd.read_csv('Book62-hashed.csv', usecols=['username'])              # main-data/
cm = dict(zip(nodes['username'], nodes['community']))
size = nodes.groupby('community').size()
G = nx.Graph(); G.add_nodes_from(nodes['username']); G.add_edges_from(zip(edges['source'], edges['target']))
lcc = max(nx.connected_components(G), key=len)
lcc_comms = sorted(set(nodes.loc[nodes['username'].isin(lcc), 'community']))
print(f'nodes {len(nodes):,} | communities {len(size):,} | largest connected component {len(lcc):,} nodes, {len(lcc_comms)} communities')
print(roles['structural_role'].value_counts().to_string())

nodes 11,426 | communities 1,667 | largest connected component 7,137 nodes, 52 communities
structural_role
Local Amplifier     6163
Attention Sink      4253
Diffusion Driver    1010


In [2]:
# (1) Concentration of Diffusion Drivers across communities
dd = roles[roles['structural_role'] == 'Diffusion Driver']
ddc = dd.groupby('community').size().reindex(size.index, fill_value=0)
print(f'Diffusion Drivers {len(dd):,}, all in the largest component: {dd.username.isin(lcc).all()} | '
      f'communities hosting at least one: {(ddc > 0).sum()} of {len(lcc_comms)}')
lc = pd.DataFrame({'size': size[lcc_comms], 'dd': ddc[lcc_comms]}); lc['rate'] = lc['dd'] / lc['size']
rho, p = stats.spearmanr(lc['size'], lc['dd'])
q1, q3 = lc['rate'].quantile([.25, .75])
print(f'DD count vs community size: Spearman rho = {rho:.2f} (p = {p:.1e}) | DD share of members: median {100*lc.rate.median():.1f}%, IQR {100*q1:.1f} to {100*q3:.1f}%')
top = ddc.sort_values(ascending=False).head(10)
print(f'ten communities with most DDs: {100*top.sum()/len(dd):.1f}% of DDs, {100*size[top.index].sum()/len(lcc):.1f}% of largest-component nodes')

Diffusion Drivers 1,010, all in the largest component: True | communities hosting at least one: 49 of 52
DD count vs community size: Spearman rho = 0.95 (p = 4.2e-26) | DD share of members: median 13.0%, IQR 8.5 to 17.3%
ten communities with most DDs: 61.3% of DDs, 55.1% of largest-component nodes


In [3]:
# 95% CI for the Spearman correlation between Diffusion Driver count and community size (Fisher z)
n = len(lc)
lo, hi = np.tanh(np.arctanh(rho) + np.array([-1.96, 1.96]) / np.sqrt(n - 3))
print(f'DD count vs community size ({n} communities): Spearman rho = {rho:.3f}, 95% CI [{lo:.3f}, {hi:.3f}] (Fisher z)')

DD count vs community size (52 communities): Spearman rho = 0.946, 95% CI [0.907, 0.969] (Fisher z)


In [4]:
# (2) Community-level brokerage: betweenness on the community graph (Table S15)
e = edges.assign(sc=edges['source'].map(cm), tc=edges['target'].map(cm))
x = e[e['sc'] != e['tc']]
M = nx.Graph()
for (a, b), w in x.groupby(['sc', 'tc'])['weight'].sum().items():
    M.add_edge(a, b, weight=M[a][b]['weight'] + w if M.has_edge(a, b) else w)
btw = pd.Series(nx.betweenness_centrality(M, normalized=True)).sort_values(ascending=False)
print(f'community graph: {M.number_of_nodes()} communities, {M.number_of_edges()} linked pairs | '
      f'top-10 share of total betweenness {100*btw.head(10).sum()/btw.sum():.1f}%')
en = {'Politik & Tuntutan': 'Politics & Demands', 'Kekerasan Aparat': 'State Apparatus Violence', 'Gerakan/Hashtag': 'Movement Coordination',
      'Ekonomi Rakyat': 'Economic Grievance', 'Demo & DPR': 'Parliament & Protest', 'Affan Kurniawan': 'Affan Kurniawan', 'Keamanan & Respons': 'Security & Response'}
tw = tweets.dropna(subset=['community']).astype({'community': int})
rows = []
for c, b in btw.head(10).items():
    nd = nodes[nodes['community'] == c]; nar = tw.loc[tw['community'] == c, 'narrative'].dropna().value_counts()
    rows.append({'Community': c, 'Betweenness': round(b, 3), 'Size': len(nd),
                 'Grassroots (%)': round(100 * (nd.account_type == 'grassroot').mean(), 1),
                 'Media / Elite': f"{(nd.account_type == 'media/news').sum()} / {(nd.account_type == 'elite/political').sum()}",
                 'Diffusion Drivers': int(ddc[c]), 'Linked Communities': M.degree(c),
                 'Dominant Narrative': f'{en[nar.index[0]]} ({100*nar.iloc[0]/nar.sum():.0f}%)'})
s15 = pd.DataFrame(rows); pd.set_option('display.width', 220); print(s15.to_string(index=False))
sz = s15['Size'].sum(); g = sum(r['Size'] * r['Grassroots (%)'] for r in rows) / sz
print(f'top-10 broker communities: {g:.1f}% grassroots, {100*s15["Diffusion Drivers"].sum()/len(dd):.1f}% of Diffusion Drivers, '
      f'Affan Kurniawan dominant in {sum("Affan" in r["Dominant Narrative"] for r in rows)} of them')

community graph: 52 communities, 328 linked pairs | top-10 share of total betweenness 94.2%
 Community  Betweenness  Size  Grassroots (%) Media / Elite  Diffusion Drivers  Linked Communities             Dominant Narrative
         2        0.459  1075            94.3       11 / 50                169                  46       Politics & Demands (33%)
         7        0.080   581            98.5         9 / 0                 54                  31 State Apparatus Violence (40%)
         9        0.080   459            95.6        11 / 9                 82                  33    Movement Coordination (37%)
        12        0.054   433            97.5         4 / 7                 60                  30 State Apparatus Violence (36%)
        11        0.052   346            97.4         7 / 2                 60                  29 State Apparatus Violence (30%)
         3        0.024   333            97.9         5 / 2                 43                  30    Movement Coordination (34%

In [5]:
# (3) Target-only accounts and the roles of accounts that posted
posted = set(authors['username'].astype(str).str.strip().str.lstrip('@').str.lower())
roles['posted'] = roles['username'].astype(str).str.strip().str.lower().isin(posted)
t = pd.crosstab(roles['account_type'], roles['posted']).rename(columns={False: 'target-only', True: 'posted'})
t['target-only (%)'] = (100 * t['target-only'] / t.sum(axis=1)).round(1); print(t)
print('roles of target-only nodes:', roles.loc[~roles['posted'], 'structural_role'].value_counts().to_dict(),
      '(3 accounts whose usernames do not match the tweet file are counted here; the manuscript reports 3,840 target-only nodes)')
p = roles[roles['posted']]
ct = pd.crosstab(p['account_type'], p['structural_role'])
print((100 * ct.div(ct.sum(axis=1), axis=0)).round(1).assign(n=ct.sum(axis=1)))
print(f"grassroots share among posting accounts {100*(p.account_type=='grassroot').mean():.1f}% | "
      f"among posting Diffusion Drivers {100*(p.loc[p.structural_role=='Diffusion Driver','account_type']=='grassroot').mean():.1f}%")

posted           target-only  posted  target-only (%)
account_type                                         
elite/political           86      31             73.5
grassroot               3699    7490             33.1
media/news                58      62             48.3
roles of target-only nodes: {'Attention Sink': 3840, 'Local Amplifier': 3} (3 accounts whose usernames do not match the tweet file are counted here; the manuscript reports 3,840 target-only nodes)
structural_role  Attention Sink  Diffusion Driver  Local Amplifier     n
account_type                                                            
elite/political            51.6              19.4             29.0    31
grassroot                   4.6              13.4             82.0  7490
media/news                 80.6               3.2             16.1    62
grassroots share among posting accounts 98.8% | among posting Diffusion Drivers 99.2%
